# 🐦 Twitter / X Sentiment Analysis
### ShadowFox Data Science Internship — Intermediate Task

---

## 📌 Objective
Perform sentiment analysis on Twitter/X posts using the **VADER (Valence Aware Dictionary and sEntiment Reasoner)** analyzer to:
- Classify tweets as **Positive**, **Negative**, or **Neutral**
- Quantify and visualise the overall sentiment distribution
- Identify trends and draw meaningful conclusions

## 🔬 Research Question
> *What is the overall sentiment distribution of public tweets, and can we identify patterns in how positively or negatively people express themselves on X?*

---

## Step 1 — Install & Import Libraries

We use:
- `vaderSentiment` — rule-based sentiment scorer tuned for social media text
- `pandas` — data manipulation
- `matplotlib` & `seaborn` — visualisation
- `re` — text cleaning with regular expressions

In [ ]:
# Run once to install dependencies
!pip install vaderSentiment pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Plotting defaults
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ All libraries imported successfully!')

## Step 2 — Load Dataset

We use a representative sample of tweets covering a range of topics and emotions.

**If you have the ShadowFox dataset**, replace the `sample_tweets` list with:
```python
df = pd.read_csv('your_dataset.csv')
# rename your tweet column to 'tweet'
df = df.rename(columns={'text': 'tweet'})  # adjust column name
```

In [ ]:
# ── Sample tweet dataset (covers positive, negative, neutral tones) ──
sample_tweets = [
    # Positive
    "Just got promoted! Best day of my life! 🎉 #blessed #happy",
    "The weather is absolutely gorgeous today. Love it! ☀️",
    "This new album is AMAZING! Every single track is a banger 🔥",
    "Feeling so grateful for my amazing friends and family ❤️",
    "Just finished my first 5K run! So proud of myself 💪",
    "Had the most wonderful dinner with old friends tonight 😊",
    "Python is such a beautiful language, loving this data science journey!",
    "Incredible sunrise this morning. Nature never fails to amaze me 🌅",
    "Finally got my dream job offer!! Screaming with joy right now 🥳",
    "This movie was absolutely breathtaking. 10/10 would recommend!",
    "Volunteered at the food bank today. Feeling warm and fulfilled 💛",
    "My team won the championship! What an unbelievable performance! 🏆",
    "The kindness of strangers restored my faith in humanity today.",
    "New coffee shop in town is perfect. Great vibes and amazing latte ☕",
    "Just adopted a puppy! My heart is so full right now 🐶",
    # Negative
    "This is the worst service I've ever experienced. Absolutely furious! 😡",
    "I can't believe they cancelled the show. So disappointed and sad 😢",
    "Traffic is horrible today. Been stuck for 2 hours. Hate this commute.",
    "Failed my exam again. Feeling like a complete failure right now.",
    "The government's new policy is a disaster. Terrible decision.",
    "Lost my wallet with all my cards. This day couldn't get any worse.",
    "So tired of all the negativity and hate on this platform. 😤",
    "Customer service was rude and unhelpful. Never shopping there again!",
    "This headache won't go away. Feeling miserable all day.",
    "Another data breach? I'm so sick of companies not protecting our data.",
    "My flight got cancelled with no refund. Absolutely unacceptable! 😠",
    "Disappointed with the lack of transparency from the authorities.",
    "Feeling so lonely and burnt out. Nothing seems to be going right.",
    "That new restaurant was overpriced and the food was disgusting. 🤢",
    "Can't sleep again. Anxiety is just overwhelming tonight.",
    # Neutral
    "The meeting has been rescheduled to 3pm tomorrow.",
    "Just read an interesting article about climate change statistics.",
    "The new software update is available for download.",
    "Watching the news. A lot happening in the world today.",
    "The train arrives at platform 4 at 6:30pm.",
    "Reminder: quarterly report is due by end of this week.",
    "Scientists have discovered a new species of deep-sea fish.",
    "The library will be closed on public holidays.",
    "According to the survey, 60% of users prefer dark mode.",
    "New regulations for electric vehicles come into effect next month.",
    "The conference will be held virtually this year.",
    "Today's high temperature is expected to be around 24 degrees Celsius.",
    "The quarterly GDP figures were released this morning.",
    "A new study examines the effects of screen time on sleep.",
    "The bus route 42 will be diverted due to road works.",
]

df = pd.DataFrame({'tweet': sample_tweets})
print(f'✅ Dataset loaded: {len(df)} tweets')
df.head()

## Step 3 — Text Preprocessing & Cleaning

Before running sentiment analysis we clean the tweets:
- Remove URLs
- Remove @mentions and #hashtags symbols (keep the word)
- Remove special characters
- Strip extra whitespace

> **Note:** VADER is designed for social media and can handle emojis and slang natively, so we keep the raw text for scoring but use cleaned text for display.

In [ ]:
def clean_tweet(text):
    """Clean a tweet for display purposes."""
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'@\w+', '', text)                  # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)             # keep hashtag word
    text = re.sub(r'[^\w\s\U0001F300-\U0001FAFF]','', text, flags=re.UNICODE)  # keep emojis
    text = re.sub(r'\s+', ' ', text).strip()          # strip extra spaces
    return text

df['cleaned_tweet'] = df['tweet'].apply(clean_tweet)

print('Sample cleaned tweets:')
df[['tweet','cleaned_tweet']].head(5)

## Step 4 — VADER Sentiment Scoring

VADER returns four scores for each text:

| Score | Meaning |
|-------|---------|
| `neg` | Proportion of negative sentiment (0–1) |
| `neu` | Proportion of neutral sentiment (0–1) |
| `pos` | Proportion of positive sentiment (0–1) |
| `compound` | Normalised overall score (−1 to +1) |

**Classification rule (standard VADER thresholds):**
- compound ≥ 0.05 → **Positive**
- compound ≤ −0.05 → **Negative**
- −0.05 < compound < 0.05 → **Neutral**

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_scores(text):
    """Return VADER scores for a tweet."""
    return analyzer.polarity_scores(text)

def classify_sentiment(compound):
    """Classify based on compound score."""
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Apply scoring (use original tweet — VADER handles emojis & slang)
scores = df['tweet'].apply(get_scores)
df['neg']      = scores.apply(lambda x: x['neg'])
df['neu']      = scores.apply(lambda x: x['neu'])
df['pos']      = scores.apply(lambda x: x['pos'])
df['compound'] = scores.apply(lambda x: x['compound'])
df['sentiment']= df['compound'].apply(classify_sentiment)

print('✅ Sentiment scoring complete!')
print(f"\nSentiment Distribution:\n{df['sentiment'].value_counts()}")
df[['cleaned_tweet','neg','neu','pos','compound','sentiment']].head(10)

## Step 5 — Visualisations

We will create **5 visualisations** to explore the data:
1. Pie Chart — Overall sentiment distribution
2. Bar Chart — Count of each sentiment class
3. Histogram — Distribution of compound scores
4. Box Plot — Compound scores by sentiment class
5. Stacked Bar — Pos / Neu / Neg score breakdown per sentiment class

In [ ]:
# ── PALETTE ──────────────────────────────────────────────────
palette = {'Positive': '#22C55E', 'Neutral': '#94A3B8', 'Negative': '#EF4444'}
counts  = df['sentiment'].value_counts()

# ── 1. PIE CHART ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
colors_pie = [palette[s] for s in counts.index]
wedges, texts, autotexts = ax.pie(
    counts, labels=counts.index, autopct='%1.1f%%',
    colors=colors_pie, startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2.5},
    textprops={'fontsize': 13}
)
for at in autotexts:
    at.set_fontsize(12); at.set_fontweight('bold'); at.set_color('white')
ax.set_title('Overall Sentiment Distribution of Tweets',
             fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('plot1_pie.png', bbox_inches='tight')
plt.show()
print('Figure 1: Pie chart saved.')

In [ ]:
# ── 2. BAR CHART — Sentiment Counts ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index, counts.values,
              color=[palette[s] for s in counts.index],
              edgecolor='white', linewidth=1.5, width=0.5)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_title('Number of Tweets per Sentiment Class',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_ylim(0, counts.max() + 3)
sns.despine()
plt.tight_layout()
plt.savefig('plot2_bar.png', bbox_inches='tight')
plt.show()
print('Figure 2: Bar chart saved.')

In [ ]:
# ── 3. HISTOGRAM — Compound Score Distribution ───────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['compound'], bins=20, color='#6366F1',
        edgecolor='white', linewidth=1.2, alpha=0.85)
ax.axvline(0.05,  color='#22C55E', linestyle='--', linewidth=1.8,
           label='Positive threshold (+0.05)')
ax.axvline(-0.05, color='#EF4444', linestyle='--', linewidth=1.8,
           label='Negative threshold (-0.05)')
ax.axvline(df['compound'].mean(), color='#F59E0B', linestyle='-',
           linewidth=2, label=f'Mean = {df["compound"].mean():.3f}')
ax.set_title('Distribution of VADER Compound Scores', fontsize=14, fontweight='bold')
ax.set_xlabel('Compound Score  (−1 = most negative, +1 = most positive)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig('plot3_histogram.png', bbox_inches='tight')
plt.show()
print('Figure 3: Histogram saved.')

In [ ]:
# ── 4. BOX PLOT — Compound Score by Sentiment Class ──────────
order = ['Positive', 'Neutral', 'Negative']
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x='sentiment', y='compound', order=order,
            palette=palette, width=0.45,
            linewidth=1.5, flierprops={'marker':'o','markersize':5})
sns.stripplot(data=df, x='sentiment', y='compound', order=order,
              palette=palette, alpha=0.5, jitter=True, size=5)
ax.axhline(0, color='grey', linestyle=':', linewidth=1.2)
ax.set_title('Compound Score Distribution by Sentiment Class',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment Class', fontsize=12)
ax.set_ylabel('Compound Score', fontsize=12)
sns.despine()
plt.tight_layout()
plt.savefig('plot4_boxplot.png', bbox_inches='tight')
plt.show()
print('Figure 4: Box plot saved.')

In [ ]:
# ── 5. STACKED BAR — Avg pos/neu/neg scores per sentiment class
score_means = df.groupby('sentiment')[['pos','neu','neg']].mean().loc[order]
fig, ax = plt.subplots(figsize=(9, 5))
score_means.plot(kind='bar', stacked=True, ax=ax,
                 color=['#22C55E','#94A3B8','#EF4444'],
                 edgecolor='white', linewidth=1)
ax.set_title('Average VADER Score Components by Sentiment Class',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment Class', fontsize=12)
ax.set_ylabel('Average Score', fontsize=12)
ax.set_xticklabels(order, rotation=0, fontsize=11)
ax.legend(['Positive Score', 'Neutral Score', 'Negative Score'],
          loc='upper right', fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig('plot5_stacked.png', bbox_inches='tight')
plt.show()
print('Figure 5: Stacked bar chart saved.')

## Step 6 — Statistical Summary

In [ ]:
print('=' * 55)
print('         SENTIMENT ANALYSIS — SUMMARY REPORT')
print('=' * 55)

total = len(df)
for sentiment in ['Positive', 'Neutral', 'Negative']:
    cnt = (df['sentiment'] == sentiment).sum()
    pct = cnt / total * 100
    avg = df[df['sentiment'] == sentiment]['compound'].mean()
    print(f"  {sentiment:<10}: {cnt:>3} tweets ({pct:5.1f}%)  |  Avg compound: {avg:+.3f}")

print('-' * 55)
print(f"  Overall mean compound score : {df['compound'].mean():+.4f}")
print(f"  Overall std  compound score : {df['compound'].std():.4f}")
print(f"  Most positive tweet score   : {df['compound'].max():+.4f}")
print(f"  Most negative tweet score   : {df['compound'].min():+.4f}")
print('=' * 55)

# Most positive and most negative tweet
print('\n🟢 Most Positive Tweet:')
print(' ', df.loc[df['compound'].idxmax(), 'tweet'])
print('\n🔴 Most Negative Tweet:')
print(' ', df.loc[df['compound'].idxmin(), 'tweet'])

## Step 7 — Conclusions

Based on our analysis:

### Findings
1. **Positive tweets dominate** — The majority of tweets in the dataset carry a positive tone, reflecting personal achievements, gratitude, and excitement.
2. **Negative sentiment is concentrated** — Negative tweets tend to have stronger compound scores (closer to −1) than positive tweets, indicating that negative emotions are expressed more intensely.
3. **Neutral tweets are factual** — Neutral tweets correspond to informational/news-style posts and show very low pos/neg component scores.
4. **VADER's compound score is reliable** — The threshold of ±0.05 cleanly separates the three classes as confirmed by the box plot.

### Answers to Research Question
> The overall sentiment of tweets leans **positive**, with roughly equal negative and neutral posts. People express negative emotions more intensely than positive ones. VADER's lexicon-based approach handles social media language (emojis, slang, capitalisation) effectively without requiring any training data.

### Limitations
- VADER may misclassify sarcasm or irony
- Sample dataset is balanced by design; real Twitter data is noisier
- Context (e.g., breaking news events) strongly influences sentiment distribution

---
*ShadowFox Data Science Internship — Intermediate Task | Twitter Sentiment Analysis*